# Modeling and Evaluation
This notebook trains and evaluates models using the saved splits and unified preprocessor. It runs the simple `scripts/run_regression.py` evaluation and also includes cells to run Ridge (with CV) and RandomForest for comparison.

In [ ]:
# Run existing regression evaluator (uses train/val/test splits)
import subprocess, os
py = os.path.join('.venv', 'Scripts', 'python.exe')
subprocess.run([py, 'scripts/run_regression.py'], check=True)

In [ ]:
# Optional: Run Ridge with 5-fold CV per-version (example)
import joblib, math, numpy as np, pandas as pd
from pathlib import Path
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error

MODELS = Path('models')
SPLITS = Path('data/processed/splits')
VERSIONS = ['1y','2y','3y']

def load_and_transform(version):
    jb = joblib.load(MODELS / f'preprocessor_{version}.joblib')
    pre = jb['preprocessor']
    freq_maps = jb.get('freq_maps', {})
    fit_columns = jb.get('fit_columns')
    id_like = jb.get('id_like', [])
    df = pd.read_csv(SPLITS / f'ensia_{version}_train.csv')
    target = jb.get('target_column')
    if target not in df.columns:
        target = [c for c in df.columns if 'average' in c.lower()][0]
    y = df[target]
    Xdf = df.drop(columns=[target] + [c for c in id_like if c in df.columns], errors='ignore')
    for c, fmap in (freq_maps or {}).items():
        if c in Xdf.columns:
            Xdf[c + '_freq'] = Xdf[c].map(fmap).fillna(0.0)
            Xdf = Xdf.drop(columns=[c])
    if fit_columns:
        Xdf = Xdf.reindex(columns=fit_columns, fill_value=np.nan)
    X = pre.transform(Xdf)
    return X, y

# Example run for 1Y (adjust as needed)
X, y = load_and_transform('1y')
cv = KFold(n_splits=5, shuffle=True, random_state=42)
alphas = [0.1, 1.0, 10.0]
ridge = RidgeCV(alphas=alphas, cv=cv, scoring='neg_mean_absolute_error')
ridge_scores = cross_val_score(ridge, X, y, cv=cv, scoring='neg_mean_absolute_error')
print('Ridge CV MAE (neg):', ridge_scores.mean(), 'std:', ridge_scores.std())

rf = RandomForestRegressor(n_estimators=50, random_state=42)
rf_scores = cross_val_score(rf, X, y, cv=cv, scoring='neg_mean_absolute_error')
print('RF CV MAE (neg):', rf_scores.mean(), 'std:', rf_scores.std())

Notes and next steps:
- For very small cohorts (3Y), prefer pooled training or careful CV.
- Use Ridge/Lasso to reduce overfitting, and tune hyperparameters.
- If you want, I can fill these notebooks with additional cells for plotting residuals and feature importance.

## Pooled training
Train on the concatenation of all `train` splits (1Y+2Y+3Y) and evaluate per-version test sets.
This helps with very small cohorts by sharing signal across years. We still evaluate per-year to see cohort-specific performance.

In [ ]:
# Pooled training: fit on all train splits, evaluate per-version tests
import joblib, math, numpy as np, pandas as pd
from pathlib import Path
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

MODELS = Path('models')
SPLITS = Path('data/processed/splits')
VERSIONS = ['1y','2y','3y']

# load a representative preprocessor metadata (first available)
meta = None
for v in VERSIONS:
    p = MODELS / f'preprocessor_{v}.joblib'
    if p.exists():
        meta = joblib.load(p)
        break
if meta is None:
    raise SystemExit('No preprocessor joblib found in models/')
pre = meta['preprocessor']
freq_maps = meta.get('freq_maps', {})
fit_columns = meta.get('fit_columns')
id_like = meta.get('id_like', [])

def prepare_Xy_from_df(df):
    target = meta.get('target_column')
    if target not in df.columns:
        possible = [c for c in df.columns if ('average' in c.lower() or 'avg' in c.lower() or 'mean' in c.lower())]
        target = possible[0] if possible else None
    if target is None:
        raise ValueError('No target column found in dataframe')
    y = df[target].values
    Xdf = df.drop(columns=[target] + [c for c in id_like if c in df.columns], errors='ignore')
    # apply frequency maps if present
    for c, fmap in (freq_maps or {}).items():
        if c in Xdf.columns:
            Xdf[c + '_freq'] = Xdf[c].map(fmap).fillna(0.0)
            Xdf = Xdf.drop(columns=[c])
    if fit_columns:
        Xdf = Xdf.reindex(columns=fit_columns, fill_value=np.nan)
    X = pre.transform(Xdf)
    return X, y

# collect pooled training data
X_list = []
y_list = []
for v in VERSIONS:
    p = SPLITS / f'ensia_{v}_train.csv'
    if not p.exists():
        continue
    df = pd.read_csv(p)
    try:
        Xv, yv = prepare_Xy_from_df(df)
    except Exception as e:
        print('Skip', v, 'train: preparation error', e)
        continue
    X_list.append(Xv)
    y_list.append(yv)

if not X_list:
    raise SystemExit('No training data available for pooled training')

X_all = np.vstack(X_list)
y_all = np.concatenate(y_list)
print('Pooled training samples:', X_all.shape[0], 'features:', X_all.shape[1])

# Fit Ridge (with small CV) and RandomForest
ridge = RidgeCV(alphas=[0.1, 1.0, 10.0], cv=5, scoring='neg_mean_absolute_error')
ridge.fit(X_all, y_all)
print('Ridge chosen alpha:', ridge.alpha_)
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_all, y_all)

# Evaluate per-version test sets
def eval_model(name, model, Xs, ys):
    y_pred = model.predict(Xs)
    mae = mean_absolute_error(ys, y_pred)
    rmse = math.sqrt(mean_squared_error(ys, y_pred))
    return mae, rmse

print('
Per-version test results (Ridge):')
for v in VERSIONS:
    p_test = SPLITS / f'ensia_{v}_test.csv'
    if not p_test.exists():
        continue
    df_test = pd.read_csv(p_test)
    try:
        Xt, yt = prepare_Xy_from_df(df_test)
    except Exception as e:
        print('Skip', v, 'test: prep error', e)
        continue
    mae, rmse = eval_model(v, ridge, Xt, yt)
    print(v.upper(), 'Ridge -> MAE={:.4f}, RMSE={:.4f}, n_test={}'.format(mae, rmse, len(yt)))

print('
Per-version test results (RandomForest):')
for v in VERSIONS:
    p_test = SPLITS / f'ensia_{v}_test.csv'
    if not p_test.exists():
        continue
    df_test = pd.read_csv(p_test)
    try:
        Xt, yt = prepare_Xy_from_df(df_test)
    except Exception as e:
        print('Skip', v, 'test: prep error', e)
        continue
    mae, rmse = eval_model(v, rf, Xt, yt)
    print(v.upper(), 'RF -> MAE={:.4f}, RMSE={:.4f}, n_test={}'.format(mae, rmse, len(yt)))